# Module 5: LangSmith Engine — Close the Loop Automatically

> *"Your proactive agent engineer."* — LangSmith

> Part of the **Modular Workshops** series. Standalone, ~30 min. **Capstone** — deploy a small agent with a deliberate bug, then watch Engine find and fix it.

In Module 4 you built the improvement loop **by hand**: trace → online eval scores a run → a run rule routes low scores to an annotation queue → a human reviews → fixes feed the next dataset.

**LangSmith Engine automates that entire loop** — and goes further. It reads your *deployed* agent's traces and runs five stages on its own:

1. **Detect** — find a *recurring* failure across many traces
2. **Diagnose** — trace the root cause into your connected GitHub source
3. **Propose a fix** — open a pull request (code or prompt)
4. **Deploy an evaluator** — add an online check so the issue can't silently regress
5. **Auto-reopen** — if the issue resurfaces after being resolved, reopen it

Engine's analysis runs automatically in the background — the first pass takes ~20 minutes, then it rescans every ~6h. You'll seed a few traces from `engine-agent` (a tiny agent that ships with an obvious bug) and turn Engine on, then walk through the issue it found, the fix it proposes, and the evaluator it deploys.

<img src="../images/engine_issue_view.png" style="width: auto; max-height: 360px; border-radius: 8px;">

## Setup

This module uses **`engine-agent`** — a tiny calculator agent in `agents/engine_agent/` that ships with a deliberate, obvious bug. It's registered in `langgraph.json` alongside the Module 3 agent, so `langgraph deploy` ships both; Engine reads its production traces. Set the deployment URL in `.env` as `LANGSMITH_DEPLOYMENT_URL` (or paste it into the seed cell below).

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os
from langsmith import Client

client = Client()
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
print("Project:", project_name)
print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))

Project: modular-workshops
LANGSMITH_API_KEY set: True


### Seed a few traces

This cell asks `engine-agent` a handful of arithmetic questions via the LangGraph SDK, giving Engine real traces to analyze (it needs only a couple to open an issue).

The catch: the agent's `calculator` tool is buggy — its **multiply** branch returns the *sum* instead of the product (`agents/engine_agent/agent.py`), and the system prompt tells the agent to trust the tool. So every multiplication answer comes back wrong (6 × 7 → 13) — a clear, recurring failure Engine can cluster and trace straight to the offending line.

In [ ]:
from langgraph_sdk import get_client
import asyncio

deployment_url = os.environ.get("LANGSMITH_DEPLOYMENT_URL") or "<your-deployment-url>"
sdk = get_client(url=deployment_url, api_key=os.environ.get("LANGSMITH_API_KEY"))

math_prompts = [
    "What is 6 times 7?",
    "What is 8 times 9?",
    "Multiply 12 by 11.",
    "What's 15 times 4?",
    "Compute 7 times 8.",
]

async def seed(q):
    await sdk.runs.wait(
        None,                 # stateless run (no thread)
        "engine-agent",       # graph name from langgraph.json
        input={"messages": [{"role": "user", "content": q}]},
        metadata={"demo": "engine"},
    )
    print("seeded:", q)

# Fire all invocations concurrently so seeding finishes in one round-trip, not five.
await asyncio.gather(*(seed(q) for q in math_prompts))

print(f"\nSeeded {len(math_prompts)} traces into project: {project_name}")
print("Now turn Engine on below.")

## 1. Turn on Engine

With traces in the project, point Engine at the agent. In LangSmith:

1. Open the **Tracing Project** your traces landed in.
2. Click the **Engine** tab.
3. **Connect code repository** — connect this repo so Engine can diagnose from source and open a fix PR (the bug lives in `agents/engine_agent/agent.py`).
4. *(Optional)* **Connect context** — point Engine at any prompts, skills, or memory files the agent uses so it reasons with the same context the agent has.
5. Pick the issue category that matches **incorrect / low-quality output** — the agent returns wrong answers.
6. Start the analysis. The first pass takes ~20 min; after that Engine rescans every ~6h.

Once analysis finishes, the rest of this module walks through what it found.

## 2. Find the issue Engine filed

When the first analysis finishes, Engine's issues appear in the **Engine** tab. You can also list them from the CLI:

In [ ]:
!cd "{project_root}" && langsmith project issues list --project "{project_name}"

## 3. Review the issue

Open the issue Engine filed (something like *Calculator returns wrong results for multiplication*). Each issue has a **title**, a **severity score**, a **category**, and a **description** — read down it the way you'd review a teammate's investigation:

- **Linked Traces** — the exact production traces Engine associated with the issue (e.g. `6 × 7 → 13`); these are the runs it used to locate the problem, so you can verify the pattern yourself.
- **Proposed Fix** — **Engine writes the fix** and shows the diff with an explanation: a **context fix** (prompt/skill/memory) and/or a **code fix** (it should land on the `multiply` branch in `agents/engine_agent/agent.py`). Review it on its merits — it's generated from *your* source, so let the proposal speak for itself rather than assuming what it'll be. Open the code fix as a GitHub PR and read the diff like any other.
- **Suggested Evaluator** — an online check Engine recommends so the issue can't silently come back (you'll apply it next).

> The main flow **leaves the bug in place** so the issue persists across demos — don't merge yet. See *Close the loop* below to resolve it end to end.

## 4. Apply the suggested evaluator

A fix closes today's gap; an evaluator **prevents the issue from coming back**. The issue's **Suggested Evaluator** is an online check Engine wrote for exactly this pattern — apply it and every new trace is scored automatically, so the issue reopens on its own if the regression returns. It completes the **feedback loop from traces to fixes to evals**, and it's the same kind of online evaluator you built by hand in Module 4 with `create_run_rule(...)` — except Engine wrote it and tied it to the issue.

*Optional:* the cell below builds an equivalent check in code, if you want to see what that evaluator is doing under the hood.

In [ ]:
from utils.langsmith_rules import create_run_rule

math_judge_prompt = (
    "You score whether the assistant's arithmetic answer is correct. "
    "Recompute the arithmetic yourself. Reply with is_correct (true/false) and "
    "one sentence of comment. Mark false if the assistant's final number is wrong "
    "(e.g. it reports a sum where a product was asked for)."
)

math_judge_schema = {
    "title": "arithmetic_correct",
    "description": "Whether the assistant's arithmetic answer is correct.",
    "type": "object",
    "properties": {
        "is_correct": {"type": "boolean", "description": "True if the final answer is mathematically correct"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["is_correct", "comment"],
    "strict": True,
}

# Same helper as Module 4 -- the equivalent of the evaluator Engine suggests on the issue.
regression_rule = create_run_rule(
    client,
    project_name=project_name,
    display_name="engine-regression-arithmetic-correct",
    sampling_rate=1.0,
    filter="eq(is_root, true)",
    llm_judge_prompt=math_judge_prompt,
    llm_judge_schema=math_judge_schema,
)
print("Regression evaluator rule ID:", regression_rule["id"])
print("Open in UI:", regression_rule["url"])

## 5. Close the loop (optional)

The main flow stops here: the bug stays in the code, so the issue persists and you can reuse it across demos. To watch Engine's loop close end to end instead:

1. **Merge** Engine's code-fix PR.
2. **Redeploy** the agent (`langgraph deploy` — same command from Module 3).
3. **Re-run the seed cell.** The multiplication answers are now correct, the evaluator passes, and Engine **resolves** the issue on its next rescan.

Heads-up: this actually fixes the bug, so **to demo again you'll need to re-introduce it** — revert the merge (or change the `multiply` branch in `agents/engine_agent/agent.py` back to `a + b`) and redeploy, and Engine re-detects it.

*Good to know:* Engine keeps **running automatically in the background** — it rescans every ~6h and auto-reopens regressions on its own, can emit webhooks (issue detected / resolved / reopened), and is metered in LCUs (1 LCU = $1.50) with an Org-Admin spend limit.

## Recap — Engine vs. the hand-built loop

| Engine stage | You did this by hand in Module 4 | Engine automates |
|---|---|---|
| **Detect** | Queried traces with `list_runs` + filters | Finds *recurring* failures across traces on its own |
| **Diagnose** | Read traces, guessed the cause | Traces the cause into connected GitHub source |
| **Propose fix** | Edited code yourself | Opens a PR (code or prompt) |
| **Add an evaluator** | `create_run_rule(...)` | Suggests an evaluator you apply in one click |
| **Route to human** | Annotation queue rule | Files issues for review + recommends offline examples |
| **Re-check** | Re-ran evals manually | Rescans every ~6h, auto-reopens regressions |

Same loop you built in Module 4 — now detected, diagnosed, fixed, and guarded automatically.

**Docs:** [LangSmith Engine](https://docs.langchain.com/langsmith/engine) · [Engine webhooks](https://docs.langchain.com/langsmith/engine-webhooks)